# Vertex Monitoring Book

This notebook contains all the details for assesing the vertex performance using the output of the `VertexMonitoringAlgorithm` provided in Pandora.

Not dependent on any ROOT/LArSoft specific tool, requires only the `.root` output files from the algorithm itself. 

## Setup on EAF

To use this notebook it is required to have on EAF a `pixi` environment 

Create it using
```bash
pixi init
```

and add base requirements with 
```bash
pixi add ipykernel matplotlib numpy pandas uproot scipy iminuit hist mplhep ipywidgets jupyterlab_widgets
```

In [ ]:
import uproot

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

import mplhep as hep
import hist

from corner import hist2d

import iminuit
from iminuit import Minuit
from iminuit.cost import UnbinnedNLL, BinnedNLL, ExtendedBinnedNLL
from scipy.stats import norm, laplace, gennorm
from scipy.special import gamma

from jacobi import propagate
from uncertainties import ufloat

import glob
import pathlib
from pathlib import Path

import sqlite3 as sql

from tqdm.auto import tqdm

from __future__ import annotations
from cycler import cycler

from typing import List, Optional, Literal

Below we define some "standards"

In [ ]:

PLOTSTYLE = {
    'axes.ymargin': 0.1,
    'legend.frameon': False,
    'xaxis.labellocation': 'right',
    'yaxis.labellocation': 'top',
    
    'axes.formatter.limits': (-2, 3),

    # 2. ax.ticklabel_format(useMathText=True)
    'axes.formatter.use_mathtext': True, 

    # 3. ax.minorticks_on()
    'xtick.minor.visible': True, 
    'ytick.minor.visible': True, 

    'xtick.major.size': 6, 
    'ytick.major.size': 6, 

    'xtick.labelsize': 14, 
    'ytick.labelsize': 14, 

    'xtick.direction': 'in', 
    'ytick.direction': 'in', 

    'xtick.top': True, 
    'ytick.right': True, 

    # 2. ax.tick_params(which='minor', length=3, direction='in', right=True, top=True)
    'xtick.minor.size': 3, 
    'ytick.minor.size': 3, 

    # Note: Direction, top, and right settings automatically apply to minor ticks 
    # when set globally, but you can explicitly ensure they mirror major ticks.
    'xtick.minor.top': True, 
    'ytick.minor.right': True, 

    'axes.xmargin': 0.0, 

    'legend.title_fontsize': 16, 
    'legend.fontsize': 14, 
    'axes.labelsize': 17, 
    'axes.titlesize': 16,

    'legend.handleheight': 1, 
    'legend.handlelength': 1.2, 

    'savefig.dpi': 300, 

    'axes.prop_cycle': (
        cycler('color', ["#0d49fb", "#e6091c", "#26eb47", "#8936df", "#fec32d", "#25d7fd"]) + 
        cycler('ls', ['-', '--', '-.', ':', '-', '--'])
    ), 
    'savefig.transparent': True,
    'savefig.bbox': 'tight',
}

plt.style.use(PLOTSTYLE)

# DATA_PATH = Path('../data')
VALIDATION_PATH = Path('/exp/icarus/data/users/msotgia/VertexDL/Validation')
DATA_PATH = VALIDATION_PATH / 'VertexMonitoringData/V10_06_00_06p03/VertexMonitoring/BNB'
REPORT_PATH = VALIDATION_PATH / 'InfrastructureData/V10_06_00_06p03/VertexMonitoring/BNB'

PLOT_PATH = Path('../plots/VertexMonitoring')
LOG_PATH = Path('../logs/VertexMonitoring')

SAVE_FORMATS = ['pdf', 'png']

MU = 'isMu == 1'
E = 'isElectron == 1'

NC = 'isCC == 0'
CC = 'isCC == 1'

QE = 'isQE == 1'
RES = 'isRes == 1'
COH = 'isCoh == 1'
DIS = 'isDIS == 1'

PRESEL = 'success == 1 & trueNuEnergy >= 0.025'

## Utilities + plotting goes here

Below there are a set of standard functions used for validation of the algorithm

In [ ]:
def load_one(file, tree = 'vertices', preselection: Optional[str | List[str]] = None):
    with uproot.open(file) as f:
        names = [k for k in f.keys(cycle=False) if k.startswith(tree)]
        dfs = []
        for name in names:
            df = f[name].arrays(library="pd")
            df["tree"] = name

            if preselection:
                de = select(df, preselection)
            
            dfs.append(df)
    
    df = pd.concat(dfs, ignore_index=True)
    return df

def load(path: Path, preselection: Optional[str | List[str]] = None):
    '''Load the data from the .root files 
    in the `path`. If `preselection`,  
    return the skimmed version of the data

    @params
        path: required, Path object
            path to the data files
        preselection: str | List[str]
    '''

    dfs = []
    
    for file in (pbar := tqdm(path.glob('*.root'))):
        df = load_one(file, preselection=preselection)
        dfs.append(df)
        pbar.set_description(f'Processing {file}, this df has {df.shape = }')

    df = pd.concat(dfs, ignore_index=True)
    return df

def success_rate(data):
    return len(data[data.success == 1]) / len(data.success)

def select(df, selection):
    if not isinstance(selection, list):
        selection = [selection]
    return df.query(' & '.join(selection))

def pdf(x, p):
    '''
    PDF for the GGD function...
    p[0] = Normalization
    p[1] = Beta
    p[2] = Mu
    p[3] = Sigma
    '''
    N, β, μ, σ = p
    scale = σ * np.sqrt(gamma(1/β) / gamma(3/β))
    return N * gennorm.pdf(x, β, μ, scale)

def cdf(x, p):
    N, β, μ, σ = p
    scale = σ * np.sqrt(gamma(1/β) / gamma(3/β))
    return p[0] * gennorm.cdf(x, β, μ, scale)

def fit(n, bins, p0: tuple, limits=None, fixed=None, cdf=cdf):

    cost = ExtendedBinnedNLL(n, bins, cdf)
    m = Minuit(cost, p0)

    if fixed:
        for f in fixed:
            m.fixed[f] = fixed[f]
    
    if limits:
        for l in limits:
            m.limits[l] = limits[l]
    m.migrad()
    m.hesse()

    return m

def figure(title, title_loc='left', selection_title=None, **kw_args):

    selection_title_loc = kw_args.pop('selection_title_loc', (0.95, 0.9))
    
    fig, ax = plt.subplots(figsize=kw_args.pop('figsize', (3.5, 4)), **kw_args)

    plt.gca().set_title(title, loc=title_loc, color='gray')

    if selection_title:
        if title_loc == 'left':
            plt.gca().set_title(selection_title, loc='right')
        else:
            plt.gca().text(
                *selection_title_loc,
                selection_title,
                transform=ax.transAxes,
                ha='right',
                va='top',
                size=plt.rcParams['axes.titlesize']
            )
    
    return fig, ax

def add_hist(ax, data, bins, label, **kw_args):
    ax.set(
        xlabel=kw_args.pop('xlabel', ''),
        ylabel=kw_args.pop('ylabel', 'Entries'),
    )

    data = np.asarray(data)
    bins = np.asarray(bins)

    # Clipped copy: only for drawing, so overflow shows up in the last bin
    flow = kw_args.pop('flow', False)

    if flow:
        clipped_data = np.clip(data, bins[0], bins[-1])
    
    ax.hist((clipped_data if flow else data), bins=bins, histtype=kw_args.pop('histtype', 'step'),
            label=label, **kw_args)

    # Returned values: original data, so under/overflow is excluded
    n, edges = np.histogram(data, bins=bins,
                            weights=kw_args.get('weights'),
                            density=kw_args.get('density', False))
    return n, edges

def add_efficiency(ax, data, success, cuts, label=None, **kw_args):
    data, success, cuts = np.asarray(data), np.asarray(success).astype(bool), np.asarray(cuts)
    passed = data[success]
    
    fraction_passed = np.array([np.count_nonzero(passed < c) for c in cuts]) / len(data)

    ax.plot(cuts, fraction_passed, label=label, **kw_args)

    ax.set(
        ylabel='Fraction of events'
    )
    

def add_hist2d(ax, x, y, bins, levels, histtype: Literal['mix', 'fill', 'contour', 'scatter'] ='mix', label=None, color='C0', **kw_args):
    """2D histogram contour: either filled (fill=True) or lines only (fill=False)."""
    x, y = np.asarray(x), np.asarray(y)

    if histtype == 'mix':
        hist2d(x, y, bins=bins, levels=levels, color=color, **kw_args)
    else:
        raise NotImplementedError(f'Option {histtype=} not yet implemented')
    '''    
    H, xedges, yedges = np.histogram2d(x, y, bins=bins, **kw_args)
    H = H.T  # contour expects Z[y, x]

    if smooth:
        from scipy.ndimage import gaussian_filter
        H = gaussian_filter(H, smooth)

    Bin centers
    xc = 0.5 * (xedges[:-1] + xedges[1:])
    yc = 0.5 * (yedges[:-1] + yedges[1:])

    if fill:
        # Colormap built from the single color: transparent -> color,
        # so several datasets can be overlaid on the same axes
        r, g, b = to_rgb(color)
        cmap = LinearSegmentedColormap.from_list(
            'single', [(r, g, b, 0.0), (r, g, b, alpha)])
        ax.contourf(xc, yc, H, levels=levels, cmap=cmap)
        if label:  # empty proxy artist for the legend
            ax.fill_between([], [], color=color, alpha=alpha, label=label)
    else:
        ax.contour(xc, yc, H, levels=levels, colors=[color],
                   linestyles=linestyle, linewidths=linewidth)
        if label:
            ax.plot([], [], color=color, linestyle=linestyle,
                    linewidth=linewidth, label=label)
    return H
    '''

def add_fit(bins, m: Minuit, scale=1.0, pdf=pdf, N=100, draw_errorband=False, **kw_args):

    x = np.linspace(bins.min(), bins.max(), N)

    if draw_errorband:
        y, ycov = propagate(lambda p: pdf(x, p)/scale, m.values, m.covariance)
    else:
        y = pdf(x, m.values)/scale

    ax.plot(x, y, **kw_args)
    
    if draw_errorband:
        yerr = np.diag(ycov) ** 0.5
        ax.fill_between(x, y - yerr, y + yerr, alpha=0.5*kw_args.pop('alpha', 1), color=kw_args.get('color', 'k'))

def add_results(ax, names, m: Minuit, loc, skip=1, separator=', ',
                uncertainties=False, fmt='.2f', **kw_args):
    x, y = loc

    pairs = []
    for a, b, be in list(zip(names, m.values, m.errors))[skip:]:
        if uncertainties:
            val = f'{ufloat(b, be):L}'   # LaTeX output, e.g. 0.12 \pm 0.01
        else:
            val = f'{b:{fmt}}'
        pairs.append(f'${a} = {val}$')

    ax.text(x, y, separator.join(pairs), **kw_args)

def save(fig, where, plot_path=PLOT_PATH, ext=SAVE_FORMATS, **kw_args):
    where: Path = plot_path / where
    where.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(ext, str):
        ext = [ext]
    
    bbox_inches = kw_args.pop('bbox_inches', 'tight')

    for e in ext:
        fig.savefig(where.with_suffix(f'.{e.lstrip(".")}'),
                    bbox_inches=bbox_inches, **kw_args)

def log(what, where, log_path=LOG_PATH):
    where: Path = log_path / where
    where.parent.mkdir(parents=True, exist_ok=True)

    with open(where.with_suffix(f'.log'), 'w') as flog:
        print(what, file=flog)

Processing the data from `/data`

In [ ]:
data_nue_standard = load(DATA_PATH / 'NuE/Standard')
data_nue_dl = load(DATA_PATH / 'NuE/DL')
data_numu_standard = load(DATA_PATH / 'NuMu/Standard')
data_numu_dl = load(DATA_PATH / 'NuMu/DL')

In [ ]:
for data in [data_nue_standard, data_nue_dl, data_numu_standard, data_numu_dl]:
    print(f'{success_rate(data) = }')

In [ ]:
data_nue_standard.head()

## Selections

Here we define a list of selections to be used for plotting 

### Muon neutrino selections + NC, on BNB $\nu_\mu$ sample

In [ ]:
SELECTIONS = [
    [PRESEL, MU, CC],
    [PRESEL, MU, CC, 'nProtons == 0'],
    [PRESEL, MU, CC, 'nProtons > 0', 'nPiPlus == 0', 'nPiMinus == 0'],
    [PRESEL, MU, CC, QE],
    [PRESEL, MU, CC, QE, 'nProtons == 0'],
    [PRESEL, MU, CC, QE, 'nProtons > 0', 'nPiPlus == 0', 'nPiMinus == 0'],
    [PRESEL, NC],
    [PRESEL, NC, 'nProtons == 0'],
    [PRESEL, NC, 'nProtons > 0', 'nPiPlus == 0', 'nPiMinus == 0'],
    [PRESEL, MU, CC, COH],
    [PRESEL, MU, CC, DIS],
    [PRESEL, MU, CC, RES],
    [PRESEL, NC, DIS],
    [PRESEL, NC, RES],
]
TITLES = [
    r'$\nu_\mu$CC',
    r'$\nu_\mu$CC0p',
    r'$\nu_\mu$CC$0\pi$',
    r'$\nu_\mu$CC QE',
    r'$\nu_\mu$CC QE 0p',
    r'$\nu_\mu$CC QE $0\pi$',
    r'$\nu$NC',
    r'$\nu$NC0p',
    r'$\nu$NC$0\pi$',
    r'$\nu_\mu$CC Coh.',
    r'$\nu_\mu$CC DIS',
    r'$\nu_\mu$CC RES',
    r'$\nu$NC DIS',
    r'$\nu$NC RES',
]
TITLES_STRIPPED = [
    'NuMu_CC',
    'NuMu_CC_0p',
    'NuMu_CC_0pi',
    'NuMu_CC_QE',
    'NuMu_CC_QE_0p',
    'NuMu_CC_QE_0pi',
    'NuMu_NC',
    'NuMu_NC_0p',
    'NuMu_NC_0pi',
    'NuMu_CC_Coh',
    'NuMu_CC_DIS',
    'NuMu_CC_Res',
    'NuMu_NC_DIS',
    'NuMu_NC_Res',
]
AXES = ['x', 'y', 'z']

# This binning is wide enought for the full outliers
# BINS = np.linspace(-1500, 1500, 1000)
BINS = np.linspace(-2.5, 2.5, 100)
ALPHA = 0.75

VARIABLES = [
    'N',
    r'\beta', 
    r'\mu', 
    r'\sigma'
]

LIMITS = {
    'x0': (0, np.inf),
    'x1': (1e-8, np.inf),
    'x3': (1e-3, np.inf)
}

FIXED = {
    'x1': True
}

SKIP = 2

P0_STANDARD = (1, 1.25, 0, 2)
P0_DL = (1, 1.25, 0, 2)

DATA_STANDARD = data_numu_standard
DATA_DL = data_numu_dl

SAMPLE_NAME = r'ICARUS BNB MC $\nu_\mu$'

rLOC_STANDARD = (0.05, 0.15)
rLOC_DL = (0.95, 0.15)


for SELECTION, TITLE, TITLE_STRIPPED in zip(SELECTIONS, TITLES, TITLES_STRIPPED):
    for AXIS in AXES:

        VAR = f'd{AXIS}'
        LABEL = f'${AXIS}_\\mathrm{{reco}} - {AXIS}_\\mathrm{{true}}$ (cm)'
        SAVE_FIG = f'Monitoring/Pandora_VertexMonitoring_{TITLE_STRIPPED}_{VAR}'
        
        fig, ax = figure(SAMPLE_NAME, figsize=(5, 3.5), selection_title=TITLE, title_loc='left')
        
        data = select(DATA_STANDARD, SELECTION)
        n, bins = add_hist(ax, data[VAR], BINS, 'Standard Pandora', density=True, ls='-')
        results = fit(n, bins, P0_STANDARD, limits=LIMITS, fixed=FIXED)
        log(results, SAVE_FIG)
        add_fit(bins, results, scale=n.sum(), color='C0', ls=':', N=200, draw_errorband=False, alpha=ALPHA)
        add_results(ax, VARIABLES, results, rLOC_STANDARD, color='C0', transform=ax.transAxes, separator='\n', uncertainties=True, skip=SKIP)
        
        data = select(DATA_DL, SELECTION)
        n, bins = add_hist(ax, data[VAR], BINS, 'DL Pandora', density=True, ls='-')
        results = fit(n, bins, P0_DL, limits=LIMITS, fixed=FIXED)
        log(results, SAVE_FIG)
        add_fit(bins, results, scale=n.sum(), color='C1', ls=':', N=200, draw_errorband=False, alpha=ALPHA)
        add_results(ax, VARIABLES, results, rLOC_DL, color='C1', transform=ax.transAxes, separator='\n', uncertainties=True, ha='right', skip=SKIP)
        
        ax.legend(loc='upper left')
        
        ax.set(xlabel=LABEL, ylabel='Normalised entries')
        
        save(fig, SAVE_FIG)

In [ ]:
AXES = ['r']

BINS = 50 # [np.linspace(-2.5, 2.5, 100), np.linspace(0, 5, 100)]
ALPHA = 0.75

VARIABLES = [
    'N',
    r'\beta', 
    r'\mu', 
    r'\sigma'
]

LIMITS = {
    'x0': (0, np.inf),
    'x1': (1e-8, np.inf),
    'x3': (1e-3, np.inf)
}

FIXED = {
    'x1': True
}

SKIP = 2

P0_STANDARD = (1, 1.25, 0, 2)
P0_DL = (1, 1.25, 0, 2)

DATA_STANDARD = data_numu_standard
DATA_DL = data_numu_dl

SAMPLE_NAME = r'ICARUS BNB MC $\nu_\mu$'

rLOC_STANDARD = (0.05, 0.15)
rLOC_DL = (0.95, 0.15)


for SELECTION, TITLE, TITLE_STRIPPED in zip(SELECTIONS, TITLES, TITLES_STRIPPED):
    for AXIS in AXES:

        VAR = f'd{AXIS}'
        COMPARE = 'trueNuEnergy'
        LABEL = f'$\\delta {AXIS}^{{3D}}_\\mathrm{{reco-true}}$ (cm)'
        SAVE_FIG = f'Monitoring/Pandora_VertexMonitoring_{TITLE_STRIPPED}_{VAR}_{COMPARE}_2D'
        
        fig, ax = figure(SAMPLE_NAME, figsize=(5, 3.5), selection_title=TITLE, title_loc='left')
        
        data = select(DATA_STANDARD, SELECTION)
        add_hist2d(ax, data[VAR], data[COMPARE], BINS, color='C0', levels=[0.39, 0.68, 0.95], range=[(0, 3.5), (0, 5)], 
                   contour_kwargs=dict(linestyles=['dotted', 'dashed', 'solid'], linewidths=[1, 1.2, 1.5]))
        
        data = select(DATA_DL, SELECTION)
        add_hist2d(ax, data[VAR], data[COMPARE], BINS, color='C1', levels=[0.39, 0.68, 0.95], range=[(0, 3.5), (0, 5)], 
                   contour_kwargs=dict(linestyles=['dotted', 'dashed', 'solid'], linewidths=[1, 1.2, 1.5]))
        
        # ax.legend(loc='upper left')
        
        ax.set(xlabel=LABEL, ylabel=r'True energy $E_\nu$')
        
        save(fig, SAVE_FIG)

In [ ]:
SELECTIONS = [
    [PRESEL, MU, CC],
    [PRESEL, MU, CC, 'nProtons == 0'],
    [PRESEL, MU, CC, 'nProtons > 0', 'nPiPlus == 0', 'nPiMinus == 0'],
    [PRESEL, MU, CC, QE],
    [PRESEL, MU, CC, QE, 'nProtons == 0'],
    [PRESEL, MU, CC, QE, 'nProtons > 0', 'nPiPlus == 0', 'nPiMinus == 0'],
    [PRESEL, NC],
    [PRESEL, NC, 'nProtons == 0'],
    [PRESEL, NC, 'nProtons > 0', 'nPiPlus == 0', 'nPiMinus == 0'],
    [PRESEL, MU, CC, COH],
    [PRESEL, MU, CC, DIS],
    [PRESEL, MU, CC, RES],
    [PRESEL, NC, DIS],
    [PRESEL, NC, RES],
]
TITLES = [
    r'$\nu_\mu$CC',
    r'$\nu_\mu$CC0p',
    r'$\nu_\mu$CC$0\pi$',
    r'$\nu_\mu$CC QE',
    r'$\nu_\mu$CC QE 0p',
    r'$\nu_\mu$CC QE $0\pi$',
    r'$\nu$NC',
    r'$\nu$NC0p',
    r'$\nu$NC$0\pi$',
    r'$\nu_\mu$CC Coh.',
    r'$\nu_\mu$CC DIS',
    r'$\nu_\mu$CC RES',
    r'$\nu$NC DIS',
    r'$\nu$NC RES',
]
TITLES_STRIPPED = [
    'NuMu_CC',
    'NuMu_CC_0p',
    'NuMu_CC_0pi',
    'NuMu_CC_QE',
    'NuMu_CC_QE_0p',
    'NuMu_CC_QE_0pi',
    'NuMu_NC',
    'NuMu_NC_0p',
    'NuMu_NC_0pi',
    'NuMu_CC_Coh',
    'NuMu_CC_DIS',
    'NuMu_CC_Res',
    'NuMu_NC_DIS',
    'NuMu_NC_Res',
]
AXES = ['r']

BINS = np.linspace(0, 3.5, 1000)
ALPHA = 0.75

VARIABLES = [
    'N',
    r'\beta', 
    r'\mu', 
    r'\sigma'
]

LIMITS = {
    'x0': (0, np.inf),
    'x1': (1e-8, np.inf),
    'x3': (1e-3, np.inf)
}

FIXED = {
    'x1': True
}

SKIP = 2

P0_STANDARD = (1, 1.25, 0, 2)
P0_DL = (1, 1.25, 0, 2)

DATA_STANDARD = data_numu_standard
DATA_DL = data_numu_dl

SAMPLE_NAME = r'ICARUS BNB MC $\nu_\mu$'

rLOC_STANDARD = (0.05, 0.15)
rLOC_DL = (0.95, 0.15)


for SELECTION, TITLE, TITLE_STRIPPED in zip(SELECTIONS, TITLES, TITLES_STRIPPED):
    for AXIS in AXES:

        VAR = f'd{AXIS}'
        COMPARE = 'trueNuEnergy'
        LABEL = f'$\\delta {AXIS}^{{3D}}_\\mathrm{{reco-true}}$ (cm)'
        SAVE_FIG = f'Monitoring/Pandora_VertexMonitoring_{TITLE_STRIPPED}_{VAR}_Efficiency'
        
        fig, ax = figure(SAMPLE_NAME, figsize=(5, 3.5), selection_title=TITLE, title_loc='left')
        
        data = select(DATA_STANDARD, SELECTION)
        add_efficiency(ax, data[VAR], data['success'], BINS, 'Standard Pandora', ls='-')
        
        data = select(DATA_DL, SELECTION)
        add_efficiency(ax, data[VAR], data['success'], BINS, 'DL Pandora', ls='-')
        
        ax.legend(loc='lower right')
        
        ax.set(xlabel=LABEL)
        
        save(fig, SAVE_FIG)

### Electron neutrino selections + NC, on BNB $\nu_e$ sample

In [ ]:
SELECTIONS = [
    [PRESEL, E, CC],
    [PRESEL, E, CC, 'nProtons == 0'],
    [PRESEL, E, CC, 'nProtons > 0', 'nPiPlus == 0', 'nPiMinus == 0'],
    [PRESEL, E, CC, QE],
    [PRESEL, E, CC, QE, 'nProtons == 0'],
    [PRESEL, E, CC, QE, 'nProtons > 0', 'nPiPlus == 0', 'nPiMinus == 0'],
    [PRESEL, NC],
    [PRESEL, NC, 'nProtons == 0'],
    [PRESEL, NC, 'nProtons > 0', 'nPiPlus == 0', 'nPiMinus == 0'],
    [PRESEL, E, CC, COH],
    [PRESEL, E, CC, DIS],
    [PRESEL, E, CC, RES],
    [PRESEL, NC, DIS],
    [PRESEL, NC, RES],
    [PRESEL, E, CC, 'nProtons == 1', 'nPhotons == 1'],
    [PRESEL, NC, 'nProtons == 1', 'nPhotons == 1'],
]
TITLES = [
    r'$\nu_\mathrm{e}$CC',
    r'$\nu_\mathrm{e}$CC0p',
    r'$\nu_\mathrm{e}$CC$0\pi$',
    r'$\nu_\mathrm{e}$CC QE',
    r'$\nu_\mathrm{e}$CC QE 0p',
    r'$\nu_\mathrm{e}$CC QE $0\pi$',
    r'$\nu$NC',
    r'$\nu$NC0p',
    r'$\nu$NC$0\pi$',
    r'$\nu_\mathrm{e}$CC Coh.',
    r'$\nu_\mathrm{e}$CC DIS',
    r'$\nu_\mathrm{e}$CC RES',
    r'$\nu$NC DIS',
    r'$\nu$NC RES',
    r'$\nu_\mathrm{e}$CC1p$1\gamma$',
    r'$\nu$NC1p$1\gamma$',
]
TITLES_STRIPPED = [
    'NuE_CC',
    'NuE_CC_0p',
    'NuE_CC_0pi',
    'NuE_CC_QE',
    'NuE_CC_QE_0p',
    'NuE_CC_QE_0pi',
    'NuE_NC',
    'NuE_NC_0p',
    'NuE_NC_0pi',
    'NuE_CC_Coh',
    'NuE_CC_DIS',
    'NuE_CC_Res',
    'NuE_NC_DIS',
    'NuE_NC_Res',
    'NuE_CC_1p_1gamma',
    'NuE_NC_1p_1gamma',
]
AXES = ['x', 'y', 'z']

BINS = np.linspace(-2.5, 2.5, 100)
ALPHA = 0.75

VARIABLES = [
    'N',
    r'\beta', 
    r'\mu', 
    r'\sigma'
]

LIMITS = {
    'x0': (0, np.inf),
    'x1': (1e-8, np.inf),
    'x3': (1e-3, np.inf)
}

FIXED = {
    'x1': True
}

SKIP = 2

P0_STANDARD = (1, 1.25, 0, 2)
P0_DL = (1, 1.25, 0, 2)

DATA_STANDARD = data_nue_standard
DATA_DL = data_nue_dl

SAMPLE_NAME = r'ICARUS BNB MC $\nu_\mathrm{e}$'

rLOC_STANDARD = (0.05, 0.15)
rLOC_DL = (0.95, 0.15)


for SELECTION, TITLE, TITLE_STRIPPED in zip(SELECTIONS, TITLES, TITLES_STRIPPED):
    for AXIS in AXES:

        VAR = f'd{AXIS}'
        LABEL = f'${AXIS}_\\mathrm{{reco}} - {AXIS}_\\mathrm{{true}}$ (cm)'
        SAVE_FIG = f'Monitoring/Pandora_VertexMonitoring_{TITLE_STRIPPED}_{VAR}'
        
        fig, ax = figure(SAMPLE_NAME, figsize=(5, 3.5), selection_title=TITLE, title_loc='left')
        
        data = select(DATA_STANDARD, SELECTION)
        n, bins = add_hist(ax, data[VAR], BINS, 'Standard Pandora', density=True, ls='-')
        results = fit(n, bins, P0_STANDARD, limits=LIMITS, fixed=FIXED)
        log(results, SAVE_FIG)
        add_fit(bins, results, scale=n.sum(), color='C0', ls=':', N=200, draw_errorband=False, alpha=ALPHA)
        add_results(ax, VARIABLES, results, rLOC_STANDARD, color='C0', transform=ax.transAxes, separator='\n', uncertainties=True, skip=SKIP)
        
        data = select(DATA_DL, SELECTION)
        n, bins = add_hist(ax, data[VAR], BINS, 'DL Pandora', density=True, ls='-')
        results = fit(n, bins, P0_DL, limits=LIMITS, fixed=FIXED)
        log(results, SAVE_FIG)
        add_fit(bins, results, scale=n.sum(), color='C1', ls=':', N=200, draw_errorband=False, alpha=ALPHA)
        add_results(ax, VARIABLES, results, rLOC_DL, color='C1', transform=ax.transAxes, separator='\n', uncertainties=True, ha='right', skip=SKIP)
        
        ax.legend(loc='upper left')
        
        ax.set(xlabel=LABEL, ylabel='Normalised entries')
        
        save(fig, SAVE_FIG)

In [ ]:
AXES = ['r']

BINS = 100 # [np.linspace(-2.5, 2.5, 100), np.linspace(0, 5, 100)]
ALPHA = 0.75

VARIABLES = [
    'N',
    r'\beta', 
    r'\mu', 
    r'\sigma'
]

LIMITS = {
    'x0': (0, np.inf),
    'x1': (1e-8, np.inf),
    'x3': (1e-3, np.inf)
}

FIXED = {
    'x1': True
}

SKIP = 2

P0_STANDARD = (1, 1.25, 0, 2)
P0_DL = (1, 1.25, 0, 2)

DATA_STANDARD = data_nue_standard
DATA_DL = data_nue_dl

SAMPLE_NAME = r'ICARUS BNB MC $\nu_\mathrm{e}$'

rLOC_STANDARD = (0.05, 0.15)
rLOC_DL = (0.95, 0.15)


for SELECTION, TITLE, TITLE_STRIPPED in zip(SELECTIONS, TITLES, TITLES_STRIPPED):
    for AXIS in AXES:

        VAR = f'd{AXIS}'
        COMPARE = 'trueNuEnergy'
        LABEL = f'$\\delta {AXIS}^{{3D}}_\\mathrm{{reco-true}}$ (cm)'
        SAVE_FIG = f'Monitoring/Pandora_VertexMonitoring_{TITLE_STRIPPED}_{VAR}_{COMPARE}_2D'
        
        fig, ax = figure(SAMPLE_NAME, figsize=(5, 3.5), selection_title=TITLE, title_loc='left')
        
        data = select(DATA_STANDARD, SELECTION)
        add_hist2d(ax, data[VAR], data[COMPARE], BINS, color='C0', levels=[0.39, 0.68, 0.95], range=[(0, 3.5), (0, 5)], 
                   contour_kwargs=dict(linestyles=['dotted', 'dashed', 'solid'], linewidths=[1, 1.2, 1.5]))
        
        data = select(DATA_DL, SELECTION)
        add_hist2d(ax, data[VAR], data[COMPARE], BINS, color='C1', levels=[0.39, 0.68, 0.95], range=[(0, 3.5), (0, 5)], 
                   contour_kwargs=dict(linestyles=['dotted', 'dashed', 'solid'], linewidths=[1, 1.2, 1.5]))
        
        # ax.legend(loc='upper left')
        
        ax.set(xlabel=LABEL, ylabel=r'True energy $E_\nu$')
        
        save(fig, SAVE_FIG)

In [ ]:
AXES = ['r']

BINS = np.linspace(0, 5, 1000)
ALPHA = 0.75

VARIABLES = [
    'N',
    r'\beta', 
    r'\mu', 
    r'\sigma'
]

LIMITS = {
    'x0': (0, np.inf),
    'x1': (1e-8, np.inf),
    'x3': (1e-3, np.inf)
}

FIXED = {
    'x1': True
}

SKIP = 2

P0_STANDARD = (1, 1.25, 0, 2)
P0_DL = (1, 1.25, 0, 2)

DATA_STANDARD = data_nue_standard
DATA_DL = data_nue_dl

SAMPLE_NAME = r'ICARUS BNB MC $\nu_\mathrm{e}$'

rLOC_STANDARD = (0.05, 0.15)
rLOC_DL = (0.95, 0.15)


for SELECTION, TITLE, TITLE_STRIPPED in zip(SELECTIONS, TITLES, TITLES_STRIPPED):
    for AXIS in AXES:

        VAR = f'd{AXIS}'
        COMPARE = 'trueNuEnergy'
        LABEL = f'$\\delta {AXIS}^{{3D}}_\\mathrm{{reco-true}}$ (cm)'
        SAVE_FIG = f'Monitoring/Pandora_VertexMonitoring_{TITLE_STRIPPED}_{VAR}_Efficiency'
        
        fig, ax = figure(SAMPLE_NAME, figsize=(5, 3.5), selection_title=TITLE, title_loc='left')
        
        data = select(DATA_STANDARD, SELECTION)
        add_efficiency(ax, data[VAR], data['success'], BINS, 'Standard Pandora', ls='-')
        
        data = select(DATA_DL, SELECTION)
        add_efficiency(ax, data[VAR], data['success'], BINS, 'DL Pandora', ls='-')
        
        ax.legend(loc='lower right')
        
        ax.set(xlabel=LABEL)
        
        save(fig, SAVE_FIG)

# Monitor Infrastructure usage

This second part of the notebook aims at analysing the resource usage for the DL vertex in Pandora for ICARUS. 

This will be key for defining a standard usage for the `Stage1` processing.

In [ ]:
def query_one_table(table: str, db_file):

    connection = sql.connect(db_file)

    df = pd.read_sql_query(f'SELECT * FROM {table}', connection)

    connection.close()
    return df

def query_tables(path: Path, table: str, preselection: Optional[str | List[str]] = None):
    dfs = []
    
    for file in (pbar := tqdm(path.glob('*.db'))):
        df = query_one_table(table)
        if preselection:
            df = select(df, preselection)
            
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)
    return df

